In [1]:
%env DEBUG=10

env: DEBUG=10


In [2]:
import argparse

import argparse
from utils import load_lora_model
from representations.main import BM25Representation
from representations.common_args import add_common_args
from influence_estimation.util import tokenize_dataset
from representations.load_data import load_data_and_estimators

from representations.main import GradientRepresentation
from representations.common_args import add_common_args
from representations.main import BaseRepresentation
from representations.load_data import (   
    load_data_and_estimators)

Debug mode enabled (DEBUG=10)


In [3]:
args=[
    "--model", "./models/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr1e-05_seed42",
    "--train_dataset_name", "loris3/ftrace_date-lm_conversational",
    "--train_dataset_split", "train",
    "--test_dataset_name", "loris3/ftrace_date-lm_conversational",
    "--test_dataset_split", "train",
    "--mode", "normalized"
]

parser = argparse.ArgumentParser()
parser = add_common_args(parser)
args = parser.parse_args(args)

train_dataset, test_dataset = load_data_and_estimators(
    args.train_dataset_name,
    args.test_dataset_name,
    args.train_dataset_split,
    args.test_dataset_split
)


_, tokenizer = load_lora_model(args.model) if not args.results_only else (None, None)


tokenized_dataset = tokenize_dataset(
    test_dataset,
    tokenizer,
    num_proc=10,
    re_index=False,
) if not args.results_only else None


tokenized_texts = [
    [tokenizer.decode([tok_id], clean_up_tokenization_spaces=True).strip()
        for tok_id in doc['input_ids']
        if tok_id not in tokenizer.all_special_ids]
    for doc in tokenized_dataset
] if tokenized_dataset else None






adapter_path ./models/OLMo-2-0425-1B_ftrace_date-lm_conversational_lr1e-05_seed42/adapter_config.json


The following generation flags are not valid and may be ignored: ['output_attentions', 'output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[WARNING] Failed to load CPU gemm_4bit_forward from kernels-community: No module named 'kernels'. Please make sure you already `pip install kernels` and the kernels >= 0.11.1


Tokenizer max length: 4096


In [4]:
rep = BM25Representation(
    tokenized_docs=tokenized_texts,  
    train_dataset=train_dataset,
    train_dataset_name=args.train_dataset_name,
    train_dataset_split=args.train_dataset_split,
    test_dataset=test_dataset,
    test_dataset_name=args.test_dataset_name,
    test_dataset_split=args.test_dataset_split,
    results_only=args.results_only
)



influence_estimate_path: ./results/influence/BM25Estimator/-/ftrace_date-lm_conversational_train/ftrace_date-lm_conversational_train/estimate.parquet
dirname: ./results/influence/BM25Estimator/-/ftrace_date-lm_conversational_train/ftrace_date-lm_conversational_train
exists: True
Copying cached gradients
cache/gradients/full/BM25Estimator/-/ftrace_date-lm_conversational/train --> /tmp/cache/gradients/full/BM25Estimator/-/ftrace_date-lm_conversational/train
len(file_batches) 32


Copying files: 100%|██████████| 32/32 [00:00<00:00, 166937.47it/s]

cache/gradients/full/BM25Estimator/-/ftrace_date-lm_conversational/train --> /tmp/cache/gradients/full/BM25Estimator/-/ftrace_date-lm_conversational/train


len(file_batches) 32


bm25 loop: 100%|██████████| 6708/6708 [07:30<00:00, 14.88it/s]


self.train_dataset_name ftrace_date-lm_conversational


[INFO] Saved influence estimate to disk
[INFO] Deleted gradient_cache_dir


In [5]:
rep.get_influence_estimate()

,0,1,2,3,4,5,6,7,8,9,...,6698,6699,6700,6701,6702,6703,6704,6705,6706,6707
0,235.622920,216.960394,216.134947,215.801133,123.612568,141.323274,123.974728,123.914510,123.221831,123.541243,...,91.220218,102.857169,83.782197,96.531089,77.846399,90.998661,96.282078,88.633841,86.036132,97.762738
1,215.782948,234.042609,215.879533,215.547466,123.381057,123.328495,139.555527,123.681546,122.988867,123.308279,...,91.220218,102.857169,83.782197,96.531089,77.846399,90.998661,96.282078,88.633841,86.036132,97.762738
2,214.963143,215.879533,230.063473,214.727661,122.632858,122.584935,122.988867,122.928650,135.926131,122.555383,...,91.220218,102.857169,83.782197,96.531089,77.846399,90.998661,96.282078,88.633841,86.036132,97.762738
3,220.395935,221.330769,220.505321,238.700430,128.164277,129.657612,128.535364,128.475146,127.782467,143.556500,...,96.856402,111.536655,86.851082,101.813493,80.831224,96.075809,100.236911,93.187364,90.094135,104.330703
4,163.329126,163.857555,163.032107,163.461186,257.669260,238.923553,240.019393,239.959176,239.266497,239.585909,...,110.579152,135.434076,107.558085,126.825256,103.476368,125.020028,125.725031,115.937310,108.336642,128.560661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6703,86.137474,86.513017,86.513017,87.604510,86.205553,87.352293,86.510248,86.510248,86.510248,86.510248,...,91.860534,97.259594,83.781471,88.420141,92.713746,230.626969,98.997767,98.211211,78.576486,85.274439
6704,79.821976,80.171006,80.171006,79.760854,77.831753,77.551164,78.120345,78.120345,78.120345,78.120345,...,74.051908,87.766913,77.643701,81.495213,75.343303,82.336509,144.302613,78.917746,69.781023,77.286235
6705,112.521697,112.988872,112.988872,113.988733,111.361811,112.425693,111.749998,111.749998,111.749998,111.749998,...,115.181149,122.240432,104.886941,114.872669,111.344289,123.900476,121.067186,310.724162,110.203862,106.875178
6706,107.765014,108.215623,108.215623,108.071435,103.652017,103.504450,104.034013,104.034013,104.034013,104.034013,...,102.506823,108.528750,105.716364,103.456018,90.574963,104.551863,104.891644,111.641822,438.629151,106.936265


In [6]:
    # estimator_less = LESSEstimator(
    #     args.model,
    #     train_dataset,
    #     args.train_dataset_name,
    #     args.train_dataset_split,
    #     test_dataset,
    #     args.test_dataset_name,
    #     args.test_dataset_split,
    #     mode=args.mode,
    #     eval_mode=True # must be True so that cache/gradients not /tmp/cache... is used
    #   )


In [7]:
rep.get_influence_estimate().shape

(6708, 6708)